# Paper B — Step 1: one-time converter (WFDB / CSV  →  NumPy + metadata CSV)

**What it does**
- PTB-XL (WFDB, `records500`) → `ptbxl_X.npy` + `ptbxl_meta.csv`
- Chapman-Shaoxing → `chapman_X.npy` + `chapman_meta.csv`. Auto-detects:
  - **figshare** version (Zheng et al. 2020: `Diagnostics.xlsx` + `ECGData/*.csv`, 10,646 patients, amplitude in **µV**)
  - **PhysioNet WFDB** version (`WFDBRecords/**/JS*.hea + .mat`, SNOMED-CT codes in `#Dx:`)

**Harmonisation done here (identical code path for both datasets)**
1. Leads re-ordered **by name** (never by position) to `I, II, III, aVR, aVL, aVF, V1–V6`.
2. Amplitude converted to **mV** from the unit stated in the file/paper, then *empirically checked*
   (lead-II peak-to-peak vs PTB-XL; Einthoven II = I + III).
3. Both resampled from 500 Hz to `TARGET_FS` with the **same** `scipy.signal.resample_poly` (polyphase,
   built-in anti-alias FIR). PTB-XL's own 100 Hz files are deliberately *not* used, so both datasets go
   through the same downsampling filter.
4. Cropped/zero-padded to exactly 10 s.

Band-pass filtering and standardisation are done in the main notebook (so they can be ablated).

Output is resumable: re-running skips already-converted records.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/Research/cardio'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---------------- CONFIG ----------------
import os
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

PTBXL_DIR   = os.environ.get('PTBXL_DIR',   '/content/data/ptbxl')     # folder containing ptbxl_database.csv
CHAPMAN_DIR = os.environ.get('CHAPMAN_DIR', '/content/data/chapman')   # figshare OR PhysioNet WFDB root
OUT_DIR     = os.environ.get('CONV_OUT',    '/content/drive/MyDrive/Research/paperB_data')
TARGET_FS   = 100      # Strodthoff et al. 2021 benchmark operates at 100 Hz
DURATION_S  = 10
N_WORKERS   = 8        # threads for file reading (I/O bound)
CHAPMAN_USE_DENOISED = False  # figshare only. Keep False: we apply our OWN identical filter to both datasets.
MAX_RECORDS = int(os.environ.get('MAX_RECORDS', '0'))  # 0 = all; e.g. 500 for a quick smoke test

In [ ]:
if IN_COLAB and OUT_DIR.startswith('/content/drive'):
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:  # e.g. VS Code Colab extension without interactive auth
        print('Drive mount failed:', e); OUT_DIR = '/content/paperB_data'; print('Falling back to', OUT_DIR)
import subprocess, sys
try:
    import wfdb  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)

**Tip (speed):** reading ~45k small files directly from Google Drive is very slow. Copy the zips to
local disk first, e.g.
```
!mkdir -p /content/data && cd /content/data && unzip -q /content/drive/MyDrive/datasets/ptb-xl.zip -d ptbxl
```
Then point `PTBXL_DIR` / `CHAPMAN_DIR` at the unzipped folders (the code searches sub-folders).

In [ ]:
import ast, json, re, time
from pathlib import Path
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from scipy.signal import resample_poly
import scipy.io as sio
import wfdb

CANON = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
_NORM = {l.upper(): l for l in CANON}
L_OUT = TARGET_FS * DURATION_S
OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
UNIT_TO_MV = {'mv': 1.0, 'uv': 1e-3, 'µv': 1e-3, 'μv': 1e-3, 'v': 1e3, 'nv': 1e-6}


def find_one(root, pattern):
    hits = sorted(Path(root).rglob(pattern))
    return hits[0] if hits else None


def reorder(sig, names):
    """sig: (T, n_leads). Re-order columns BY NAME to CANON. Raises if any lead is missing."""
    idx = {}
    for j, n in enumerate(names):
        key = _NORM.get(str(n).strip().upper())
        if key is not None and key not in idx:
            idx[key] = j
    missing = [l for l in CANON if l not in idx]
    if missing:
        raise ValueError(f'missing leads {missing} in {names}')
    return sig[:, [idx[l] for l in CANON]]


def resample_fix(sig, fs_in):
    sig = np.nan_to_num(sig.astype(np.float64), nan=0.0)
    if fs_in != TARGET_FS:
        fr = Fraction(TARGET_FS, int(round(fs_in))).limit_denominator(1000)
        sig = resample_poly(sig, fr.numerator, fr.denominator, axis=0)
    if sig.shape[0] >= L_OUT:
        sig = sig[:L_OUT]
    else:
        sig = np.pad(sig, ((0, L_OUT - sig.shape[0]), (0, 0)))
    return sig.astype(np.float32)


def quality_flags(sig):
    """flat-lead count and Einthoven residual on the *converted* mV signal."""
    flat = int((sig.std(axis=0) < 1e-3).sum())
    I, II, III = sig[:, 0], sig[:, 1], sig[:, 2]
    rms = lambda v: float(np.sqrt(np.mean(v ** 2)) + 1e-9)
    ein = rms(II - I - III) / rms(II)
    aug = rms(sig[:, 3] + sig[:, 4] + sig[:, 5]) / rms(sig[:, 5])
    p2p = float(np.percentile(II, 99.5) - np.percentile(II, 0.5))
    return flat, ein, aug, p2p


def convert_all(name, n, loader, meta):
    """Generic resumable loop. loader(i) -> (sig_mV(T,12 canon order), fs) ; writes memmap + done mask."""
    xpath, dpath = OUT / f'{name}_X.npy', OUT / f'{name}_done.npy'
    if xpath.exists() and dpath.exists():
        X = np.load(xpath, mmap_mode='r+'); done = np.load(dpath)
        if X.shape != (n, L_OUT, 12):
            raise RuntimeError(f'{xpath} has shape {X.shape}, expected {(n, L_OUT, 12)}; delete it to restart')
    else:
        X = np.lib.format.open_memmap(xpath, mode='w+', dtype=np.float32, shape=(n, L_OUT, 12))
        done = np.zeros(n, dtype=np.int8)  # 0 todo, 1 ok, -1 failed
    flags = {k: np.full(n, np.nan) for k in ['flat', 'einthoven', 'aug_sum', 'p2p_II']}
    fpath = OUT / f'{name}_flags.npz'
    if fpath.exists():
        f = np.load(fpath); flags = {k: f[k].copy() for k in flags}
    errors = []

    def work(i):
        try:
            sig, fs = loader(i)
            s = resample_fix(sig, fs)
            return i, s, quality_flags(s), None
        except Exception as e:
            return i, None, None, f'{type(e).__name__}: {e}'

    todo = np.where(done == 0)[0]
    print(f'[{name}] {len(todo)} / {n} records to convert')
    t0 = time.time()
    with ThreadPoolExecutor(N_WORKERS) as ex:
        for k, (i, s, q, err) in enumerate(ex.map(work, todo)):
            if err is None:
                X[i] = s; done[i] = 1
                for key, v in zip(flags, q): flags[key][i] = v
            else:
                done[i] = -1; X[i] = 0; errors.append((int(i), err))
            if (k + 1) % 2000 == 0 or k + 1 == len(todo):
                X.flush(); np.save(dpath, done); np.savez(fpath, **flags)
                print(f'  {k + 1}/{len(todo)}  ({time.time() - t0:.0f}s)')
    X.flush(); np.save(dpath, done); np.savez(fpath, **flags)
    meta = meta.copy()
    meta['ok'] = (done == 1)
    for key in flags: meta[key] = flags[key]
    if errors:
        print(f'  {len(errors)} failures, first 5:', errors[:5])
        err_map = dict(errors)
        meta['error'] = [err_map.get(i, '') for i in range(n)]
    return meta, errors

## PTB-XL

In [ ]:
ptb_csv = find_one(PTBXL_DIR, 'ptbxl_database.csv')
assert ptb_csv is not None, f'ptbxl_database.csv not found under {PTBXL_DIR}'
PTB_ROOT = ptb_csv.parent
db = pd.read_csv(ptb_csv)
if MAX_RECORDS: db = db.iloc[:MAX_RECORDS]
db = db.reset_index(drop=True)
print('PTB-XL records:', len(db), '| patients:', db.patient_id.nunique(),
      '| (paper v1.0.1: 21,837 / 18,885 ; PhysioNet v1.0.3: 21,799 / 18,869)')


def load_ptb(i):
    sig, f = wfdb.rdsamp(str(PTB_ROOT / db.filename_hr[i]))
    units = [UNIT_TO_MV[u.strip().lower()] for u in f['units']]
    sig = sig * np.asarray(units)[None, :]
    return reorder(sig, f['sig_name']), f['fs']


ptb_meta = pd.DataFrame({
    'idx': np.arange(len(db)), 'dataset': 'ptbxl', 'record_id': db.ecg_id.astype(int).astype(str),
    'patient_id': 'ptb_' + db.patient_id.astype(int).astype(str), 'strat_fold': db.strat_fold.astype(int),
    'age': db.age, 'sex': db.sex.map({0: 'M', 1: 'F'}),   # PTB-XL: male 0, female 1 (Wagner et al. 2020)
    'scp_codes': db.scp_codes, 'heart_axis': db.get('heart_axis'),
    'validated_by_human': db.get('validated_by_human'),
    **{c: db[c].notna() for c in ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems'] if c in db},
})
ptb_meta, ptb_err = convert_all('ptbxl', len(db), load_ptb, ptb_meta)
ptb_meta.to_csv(OUT / 'ptbxl_meta.csv', index=False)

## Chapman-Shaoxing — format detection

In [ ]:
diag = find_one(CHAPMAN_DIR, 'Diagnostics.xlsx') or find_one(CHAPMAN_DIR, 'Diagnostics.csv')
heas = list(Path(CHAPMAN_DIR).rglob('*.hea'))
if diag is not None:
    CH_FORMAT = 'figshare'
elif heas:
    CH_FORMAT = 'wfdb'
else:
    raise FileNotFoundError(f'Neither Diagnostics.xlsx nor *.hea found under {CHAPMAN_DIR}')
print('Chapman format detected:', CH_FORMAT)

In [ ]:
if CH_FORMAT == 'figshare':
    dg = pd.read_excel(diag) if diag.suffix == '.xlsx' else pd.read_csv(diag)
    sub = 'ECGDataDenoised' if CHAPMAN_USE_DENOISED else 'ECGData'
    sig_dir = [p for p in Path(CHAPMAN_DIR).rglob(sub) if p.is_dir()]
    assert sig_dir, f'folder {sub}/ not found (unzip ECGData.zip first)'
    SIG_DIR = sig_dir[0]
    if MAX_RECORDS: dg = dg.iloc[:MAX_RECORDS]
    dg = dg.reset_index(drop=True)
    # Zheng et al. 2020: "The amplitude unit was microvolt." -> nominal scale 1e-3 to mV (checked empirically below)
    CH_SCALE = 1e-3

    def load_ch(i):
        df = pd.read_csv(SIG_DIR / f'{dg.FileName[i]}.csv')
        n = len(df)
        fs = 500 if n in (5000, 4999, 5001) else n / DURATION_S
        return reorder(df.values.astype(np.float64) * CH_SCALE, list(df.columns)), fs

    ch_meta = pd.DataFrame({
        'idx': np.arange(len(dg)), 'dataset': 'chapman', 'subsource': 'shaoxing',
        'record_id': dg.FileName.astype(str), 'patient_id': 'ch_' + dg.FileName.astype(str),  # 1 ECG per patient
        'age': dg.PatientAge, 'sex': dg.Gender.astype(str).str.upper().str[0].map({'M': 'M', 'F': 'F'}),
        'rhythm': dg.Rhythm.astype(str).str.strip(), 'beat': dg.Beat.fillna('NONE').astype(str).str.strip(),
        'snomed': '',
    })
else:
    rec_paths = sorted(p.with_suffix('') for p in heas)
    if MAX_RECORDS: rec_paths = rec_paths[:MAX_RECORDS]

    def parse_hea(p):
        lines = Path(str(p) + '.hea').read_text(errors='ignore').splitlines()
        head = lines[0].split()
        n_sig, fs = int(head[1]), float(head[2])
        names, gains, bases, units = [], [], [], []
        for ln in lines[1:1 + n_sig]:
            tok = ln.split()
            g = tok[2]  # e.g. "1000/mV", "1000.0(0)/mV", "1000"
            m = re.match(r'([-\d.eE]+)(?:\((-?\d+)\))?(?:/(\S+))?', g)
            gains.append(float(m.group(1)) or 200.0)  # WFDB: gain 0 means default 200
            bases.append(int(m.group(2)) if m.group(2) else 0)
            units.append(m.group(3) or 'mV')
            names.append(tok[-1])
        meta = {'Age': '', 'Sex': '', 'Dx': ''}
        for ln in lines:
            if ln.startswith('#'):
                k, _, v = ln[1:].partition(':')
                if k.strip() in meta: meta[k.strip()] = v.strip()
        return fs, names, np.array(gains), np.array(bases), units, meta

    hdr = [parse_hea(p) for p in rec_paths]

    def load_ch(i):
        fs, names, gains, bases, units, _ = hdr[i]
        val = sio.loadmat(str(rec_paths[i]) + '.mat')['val'].astype(np.float64)  # (n_sig, T)
        if val.shape[0] != len(names): val = val.T
        mv = (val - bases[:, None]) / gains[:, None] * np.array([UNIT_TO_MV[u.lower()] for u in units])[:, None]
        return reorder(mv.T, names), fs

    def _age(a):
        try: return float(a)
        except ValueError: return np.nan

    ch_meta = pd.DataFrame({
        'idx': np.arange(len(rec_paths)), 'dataset': 'chapman',
        # PhysioNet "ecg-arrhythmia" (45,152 ECGs). Reyna et al. 2022 list Chapman-Shaoxing 10,247 + Ningbo 34,905
        # = 45,152 recordings, so this version very likely mixes TWO hospitals. See report below.
        'subsource': 'shaoxing+ningbo(?)',
        'record_id': [p.name for p in rec_paths], 'patient_id': ['ch_' + p.name for p in rec_paths],
        'age': [_age(h[5]['Age']) for h in hdr],
        'sex': [str(h[5]['Sex'])[:1].upper() or np.nan for h in hdr],
        'rhythm': '', 'beat': '', 'snomed': [h[5]['Dx'].replace(' ', '') for h in hdr],
    })
    CH_SCALE = 1.0
    cn = find_one(CHAPMAN_DIR, 'ConditionNames_SNOMED-CT.csv')
    if cn is not None:
        import shutil; shutil.copy(cn, OUT / cn.name)   # main notebook maps SNOMED -> acronym with it
    else:
        print('!!! ConditionNames_SNOMED-CT.csv not found in the Chapman folder')
print(len(ch_meta), 'Chapman records')

In [ ]:
ch_meta, ch_err = convert_all('chapman', len(ch_meta), load_ch, ch_meta)

## Unit / lead sanity check (the "µV vs mV" silent bug)
Median lead-II peak-to-peak should be similar in both datasets (≈1–3 mV for resting 12-lead ECGs).
A ratio of ~1000 or ~4.88 would indicate a unit or ADC-gain error. Einthoven residual (II − I − III) ≈ 0 and
aVR + aVL + aVF ≈ 0 confirm lead naming/order (the same constraints Zheng et al. 2020 used for validation).

In [ ]:
def summarise(m):
    ok = m[m.ok]
    return {'n_ok': int(m.ok.sum()), 'n_failed': int((~m.ok).sum()),
            'median_p2p_II_mV': float(ok.p2p_II.median()), 'median_einthoven_resid': float(ok.einthoven.median()),
            'median_aug_sum_resid': float(ok.aug_sum.median()), 'records_with_flat_leads': int((ok.flat > 0).sum())}

rep = {'ptbxl': summarise(ptb_meta), 'chapman': summarise(ch_meta), 'chapman_format': CH_FORMAT,
       'chapman_scale_applied': CH_SCALE, 'target_fs': TARGET_FS,
       'SYNTHETIC': bool(find_one(PTBXL_DIR, 'SYNTHETIC_DATA_MARKER') or find_one(CHAPMAN_DIR, 'SYNTHETIC_DATA_MARKER'))}
ratio = rep['chapman']['median_p2p_II_mV'] / rep['ptbxl']['median_p2p_II_mV']
rep['p2p_ratio_chapman_over_ptbxl'] = ratio
print(json.dumps(rep, indent=2))
if not (0.5 < ratio < 2.0):
    cands = {'x1': 1.0, 'x1e-3': 1e-3, 'x4.88e-3': 4.88e-3, 'x4.88': 4.88, 'x1e3': 1e3}
    best = min(cands, key=lambda k: abs(np.log(ratio * cands[k])))
    rep['UNIT_WARNING'] = f'p2p ratio {ratio:.3g} is implausible. Suggested extra scale for Chapman: {best}'
    print('!!! UNIT WARNING !!!', rep['UNIT_WARNING'], '-> set CH_SCALE accordingly, delete chapman_* files, re-run.')
else:
    print(f'OK: amplitude ratio Chapman/PTB-XL = {ratio:.2f} (no unit bug)')
for k in ['ptbxl', 'chapman']:
    if rep[k]['median_einthoven_resid'] > 0.1:
        print(f'!!! {k}: Einthoven residual {rep[k]["median_einthoven_resid"]:.2f} > 0.1 -> lead names/order suspicious')
if CH_FORMAT == 'wfdb':
    print('NOTE: PhysioNet WFDB version: 45,152 = 10,247 (Chapman-Shaoxing) + 34,905 (Ningbo) recordings per '
          'Reyna et al. 2022 Table 2 -> external set likely spans two Zhejiang hospitals. Report it that way.')

In [ ]:
ch_meta.to_csv(OUT / 'chapman_meta.csv', index=False)
json.dump(rep, open(OUT / 'conversion_report.json', 'w'), indent=2)
for f in sorted(OUT.iterdir()):
    print(f'{f.name:28s} {f.stat().st_size / 1e6:9.1f} MB')
print('\nDONE. Main notebook reads ptbxl_X.npy, ptbxl_meta.csv, chapman_X.npy, chapman_meta.csv from', OUT)